In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/heptapod/titanic/train_and_test2.csv
/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [3]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [4]:
X = train.drop(columns=["PassengerId", "Survived", "Name", "Ticket", "Cabin"])
y = train["Survived"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

model = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(max_iter=1000))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring="accuracy")

print(scores.mean(), scores.std())

0.7923921913250894 0.01744462427171013


**Leitura do resultado**

**cv = 0.7924 (79,24)** com **desvio padrão = 0.0174 (1,74)**

- **79,24% é um bom ponto de partida.** O Titanic tem um teto prático estiamado em torno de 82-84% de accuracy pela comunidade. Baselines ingênuos ficam em torno de 61,6% (prever todo mundo com "não sobreviveu") ou 78,7% (prever todas as mulheres como sobreviventes). O modelo supera ambos, mas ainda está próximo da regra de sexo, o que indica espaço para feature engineering.
  
- **O Desvio padrão é aceitável.** O resultado varia de fold para fold ente aproximadamente 77,5% e 81%. Isso é esperado para 891 amostras, onde cada fold tem cerca de 178 exemplos e o desvio corresponde a ~3 acertos. Acima de 3% seria motivo para investigar instabilidade.

  
- **CV e LB devem ficar próximos.** Se a submissão fica entre 0,76% e 0,80, a validação está confiável Se ficar abaixo de 0,74 ou acima de 0,82, há algo a investigar (leakage, variância ou erro no pipeline).

**Conclusão prática:** a validação cruzada é confiável o suficiente para guiar as decisões das próximas etapas (feature engineering e escolha de modelos). 